# Aula Lab. 4 - OpenCV
## Transformações Espaciais/Geométricas (cont.s)
Nestes exercícios vamos entnder os conceitos de:
* Coordenadas e Transformação Reversa
    * f(x, y) = T(u,v)
* Métodos de Interpolação
    * exemplos: vizinho mais próximo (nearest neighbor) e bilinear
* Transformações afins e suas matrizes
    * Translado (translate)
    * Escala (scale)
    * Rotação (rotation)
    * Cisalhamento (shear)

Começamos com o cabeçalho padrão:

In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import math

cv.samples.addSamplesDataSearchPath("data")

## 4. Rotação
A rotação padrão é o processo de girar os vértices/pontos/pixels de um plano com relação ao eixo

![Rotação em cada um dos eixos](https://danceswithcode.net/engineeringnotes/rotations_in_3d/images/fig01.png)

Então temos:
* Rotação do plano XY pelo eixo Z
* Rotação do plano XZ pelo eixo Y
* Rotação do plano YZ pelo eixo X

Para a fórumla para rotação em XY temos:

![XY Rotation Formula](https://media.geeksforgeeks.org/wp-content/uploads/Rotation-Diagram.png)

* x = u&sdot;cos&theta; - v&sdot;sen&theta;
* y = u&sdot;sen&theta; + v&sdot;cos&theta;

Assim temos a matriz de afinidade $\begin{vmatrix}x\\y\\1\end{vmatrix}=\begin{vmatrix}cos\theta&-sin\theta&0\\sin\theta&cos\theta&0\\0&0&1\end{vmatrix}\begin{vmatrix}u\\v\\1\end{vmatrix}$, com a matriz simplificada para $\begin{vmatrix}x\\y\end{vmatrix}=\begin{vmatrix}cos\theta&-sin\theta&0\\sin\theta&cos\theta&0\end{vmatrix}\begin{vmatrix}u\\v\end{vmatrix}$

### Exercícios

#### 1. Rotação positiva e negativa

Iremos usar a imagem butterfly.jpg, em formato RGB, nestes exemplos.

**Muito importante!**

Uma rotação é difinida como um movimento saindo do eixo x em direção ao eixo y
Mas nosso eixo y é positivo para baixo, então teremos neste exercício:

* Valores de &theta; positivos irão girar a imagem no sentido do relógio (CW - clockwise)
* Valores de &theta; negativos irão girar a imagem no sentido contrátio do relógio (CCW - counter clockwise)

Nos demos exercícios, quando se utiliza _getRotationMatrix2D_ do OpenCV, ele automaticamente dará como resultado uma matriz com o angulo invertido!
Assim o movimento retornado por _getRotationMatrix2D_ é o movimento natural esperado (CCW para ângulo positivo) na imagem resultado.

> Lembrando que o valor de &theta; é expresso em radianos ao usar sin/cos de math

* Faça uma função que recebe uma imagem, e o ângulo de rotação, em graus.
* Faça um subplor 1x3
    * subplot 1: imagem original
    * subplot 2: imagem rotacionada 30&deg;
    * subplot 3: imagem rotacionada -60&deg;

> Perceba que todas as rotações estão centralizadas em (0,0)

In [ ]:
def rotate(img, angle:float = 0):
    ?
?

#### 2. Rotações especiais

Alguns angulos de rotação são espciais:
* 0&deg;: a imagem de destino é igual a de entrada
* 90&deg;: a imagem de destino inverteu linhas com colunas
* 180&deg;: a imagem de destino é o espelho vertical da original
* 270&deg;: a imagem de destino inverteu linhas com colunas espelhadas na vertical

As suas respectivas matrizes são:
| 0&deg; | 90&deg; | 180&deg; | 270&deg; |
| --- | --- | --- | --- |
| $\begin{vmatrix}1&0\\0&1\end{vmatrix}$ | $\begin{vmatrix}0&-1\\1&0\end{vmatrix}$ | $\begin{vmatrix}-1&0\\0&-1\end{vmatrix}$ | $\begin{vmatrix}0&1\\-1&0\end{vmatrix}$ |

A implementação desses angulos especiais normalmente são acelerados internamente, pois são apenas cópias de valores com índices invertidos.

* Crie as matrizes de rotação rot90, rot180 e rot270

Voce não verá nenhum resultado pois esses angulos ao redor do ponto central colocam a imgem fora da área visível

* troque as matrizes de rotação rot90, rot180 e rot270 pelo retorno de getRotationMatrix2D((l//2, c//2), ANGULO, 1.0)

**Voce ainda não verá nenhum um resultado correto por causa da imagem não ser quadrada**

Para usar as rotações de ângulo especial, utilize a função especial _rotate_ de OpenCV. Ela recebe como argumento ROTATE_90_CLOCKWISE, ROTATE_180 e ROTATE_90_COUNTERCLOCKWISE.

Ou você pode obter os mesmos efeitos usando _transpose_ (inverte x com y) e _flip_ (espelhamento horizontal)

In [ ]:
# Código com matrizes feitas à mão com np.float32
rot90 = ?
rot180 = ?
rot270 = ?

In [ ]:
# Código com matrizes feitas com getRotationMatrix2D
center = ?
rot90  = ?
rot180 = ?
rot270 = ?

In [ ]:
# Código usando cv.rotate
?

#### 3. Rotações fora da origem das coordenadas

Você deve ter notado que usamos _getRotationMatrix2D_, ela recebe o ponto de origem aonde será feita a rotação, o ângulo (em graus) e um fator de escala.

A matriz de afinidade de rotação pode incluir o deslocamento do centro de coratão. Já vemos que o translado ocupa a terceira coluna.

O calculo do translado irá levar em consideração os pontos Cx e Cy junto com a coordenada baseada no ângulo &theta;, então teremos:
* tx = Cx&sdot;(1-cos&theta;) + Cy&sdot;sin&theta;
* ty = Cy&sdot;(1-cos&theta;) - Cx&sdot;sin&theta;

$$\begin{vmatrix}x\\y\\1\end{vmatrix}=\begin{vmatrix}cos\theta&-sin\theta&C_{x}(1-cos\theta)+C_{y}sin\theta\\sin\theta&cos\theta&C_{y}(1-cos\theta)-C_{x}sin\theta\\0&0&1\end{vmatrix}\begin{vmatrix}u\\v\\1\end{vmatrix}$$

* Vamos mudar nossa imagem para uma imagem quadrada, utilize apple.jpg em RGB.
* Imprima as dimenções da imagem e o ponto central
* Monte uma matrix para rotação com ponto central em 45 graus manualmente e imprima
* Use _getRotationMatrix2D_ para o mesmo ângulo (e escala 1.0) e ponto central e imprima
* Observe qua as duas matrizes devem ser similares (fora erros de arredondamento)
* crie um subplot 1x2 com a imagem original, e a rotacionada

> lembre-se que _getRotationMatrix2D_ retorna com a direção da rotação corrigida, no seu calculo manual, use -45

In [ ]:
# seu código

## 4. Cizalhamento

Cizalhamento desloca cada uma das linhas **ou** das colunas de uma imagem de forma linear, transformando um paralelogramo em um trapezóide.

![Shearing](https://thepythoncode.com/media/articles/image-transformations-using-opencv-in-python/shearing-explained.png)

Temos então:

* x = u + Sx&sdot;v
* y = v + Sy&sdot;u

Coma a matriz de afinidade: $\begin{vmatrix}x\\y\\1\end{vmatrix}=\begin{vmatrix}1&S_{x}&0\\S_{y}&1&0\\0&0&1\end{vmatrix}\begin{vmatrix}u\\v\\1\end{vmatrix}$ e a versão simplificada $\begin{vmatrix}x\\y\end{vmatrix}=\begin{vmatrix}1&S_{x}&0\\S_{y}&1&0\end{vmatrix}\begin{vmatrix}u\\v\end{vmatrix}$

### Exercícios

#### 1. Cisalhamento Simples Horizontal/Vertical

* Abra left.jpg em RGB
* Crie uma função que recebe uma imagem, o cisalhamento em x (sx) e o cisalhamento em y (sy)
* Crie a matriz de cisalhamento manualmente com np.float32
* Faça um subplot 1x4
    * 1 imagem original
    * 2 cisalhamento em x -10%
    * 3 cisalhamento em y 10%
    * 3 cisalhamento em x -10% e em y 10%

In [ ]:
def shear(img, sx, sy):
    l,c = img.shape[:2]
    mat = np.float32([[1, sx, 0],[sy, 1, 0]])
    return cv.warpAffine(img, mat, (c, l))

left = cv.imread(cv.samples.findFile("left.jpg"), cv.IMREAD_COLOR_RGB)

plt.subplot(1, 4, 1)
plt.axis('off')
plt.imshow(left)

plt.subplot(1, 4, 2)
plt.axis('off')
plt.imshow(shear(left, -.1, 0))

plt.subplot(1, 4, 3)
plt.axis('off')
plt.imshow(shear(left, 0, .1))

plt.subplot(1, 4, 4)
plt.axis('off')
plt.imshow(shear(left, -.1, .1))

plt.gcf().set_dpi(250.0)

plt.show()

## Desafio

Faça 3 quadrados de 256x256 RGB, um vermelho um verde e um azul (dica: np.ones * cor)

Utilizando tudo que você aprendeu (translado, escala, rotação e cisalhamento) faça o cubo:

![Cubo Desafio](aula4_desafio.png)

Dicas:

1. Quando usar warpAffine, sempre o argumetno do tamanho da imagem de destino será 512x512
1. Para rotação use getRotationMatrix2D com escala maior de 1.0

In [ ]:
# as 3 imagens iniciais
red = ?
green = ?
blue = ?

plt.subplot(1, 3, 1)
plt.axis('off')
plt.imshow(red)
plt.subplot(1, 3, 2)
plt.axis('off')
plt.imshow(green)
plt.subplot(1, 3, 3)
plt.axis('off')
plt.imshow(blue)
plt.show()

In [ ]:

# as imagens transformadas
?

red2 = ?
green2 = ?
blue2 = ?

?

plt.subplot(1, 3, 1)
plt.axis('off')
plt.imshow(red2)
plt.subplot(1, 3, 2)
plt.axis('off')
plt.imshow(green2)
plt.subplot(1, 3, 3)
plt.axis('off')
plt.imshow(blue2)
plt.show()

In [ ]:
# a junção das 3 imagens
cube = red2 + green2 + blue2
plt.axis('off')
plt.imshow(cube)
plt.show()